# B2-019-attention-transformers — Practice p11 — Solution

**Type:** constrained-coding · **Difficulty:** advanced · **Concepts:** scaled-dot-product-attention

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260808`  
**Qualified Book 1 prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C11-neural-training`  
**Remediation:** review the linked Book 1 units before continuing: [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb).

## Independent solution

The module validates shapes and Boolean mask broadcastability before computing q @ k.transpose(-1,-2) / sqrt(Dk). It rejects fully invalid rows, fills forbidden scores with negative infinity before softmax, and leaves all input tensors unchanged.

In [ ]:
import math
import numpy as np
import torch
from torch import nn

SEED = 20260808
ATOL = 1e-10
RTOL = 1e-10
torch.manual_seed(SEED)

class ScaledMaskedAttention(nn.Module):
    def forward(self, q, k, v, allowed):
        if q.ndim != 3 or k.ndim != 3 or v.ndim != 3:
            raise ValueError("q, k, and v must be rank three")
        if q.shape[0] != k.shape[0] or k.shape[0] != v.shape[0]:
            raise ValueError("batch sizes must agree")
        if q.shape[-1] == 0 or q.shape[-1] != k.shape[-1]:
            raise ValueError("query and key widths must agree and be nonempty")
        if k.shape[-2] != v.shape[-2]:
            raise ValueError("key and value lengths must agree")
        if allowed.dtype != torch.bool:
            raise ValueError("allowed must be Boolean")
        if q.device.type != "cpu" or k.device.type != "cpu" or v.device.type != "cpu":
            raise ValueError("CPU tensors are required")
        try:
            mask = torch.broadcast_to(allowed, (q.shape[0], q.shape[1], k.shape[1]))
        except RuntimeError as exc:
            raise ValueError("allowed must broadcast to score shape") from exc
        if torch.any(~torch.any(mask, dim=-1)):
            raise ValueError("every query row needs at least one allowed key")
        scores = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(q.shape[-1])
        weights = torch.softmax(scores.masked_fill(~mask, float("-inf")), dim=-1)
        output = torch.matmul(weights, v)
        return weights, output

q = torch.tensor([[[1.0, 2.0], [-1.0, 0.5]]], dtype=torch.float64)
k = torch.tensor([[[2.0, 0.0], [0.0, 1.0], [1.0, -1.0]]], dtype=torch.float64)
v = torch.tensor([[[2.0, -1.0], [0.0, 3.0], [4.0, 1.0]]], dtype=torch.float64)
allowed = torch.tensor([[[True, True, False], [True, False, True]]], dtype=torch.bool)
reject_allowed = torch.tensor(
    [[[True, True, False], [False, False, False]]], dtype=torch.bool
)
q_before, k_before, v_before, allowed_before = q.clone(), k.clone(), v.clone(), allowed.clone()
q_np, k_np, v_np = q.numpy(), k.numpy(), v.numpy()
allowed_np = allowed.numpy()
scores_np = q_np @ np.swapaxes(k_np, -1, -2) / math.sqrt(q.shape[-1])
masked_np = np.where(allowed_np, scores_np, -np.inf)
shifted_np = masked_np - np.max(masked_np, axis=-1, keepdims=True)
exp_np = np.exp(shifted_np)
expected_weights = exp_np / np.sum(exp_np, axis=-1, keepdims=True)
expected_output = expected_weights @ v_np
module = ScaledMaskedAttention()
weights, output = module(q, k, v, allowed)
rejected = False
try:
    module(q, k, v, reject_allowed)
except ValueError:
    rejected = True

### Answer check

In [ ]:
assert weights.shape == (1, 2, 3) and output.shape == (1, 2, 2)
assert weights.dtype == output.dtype == torch.float64
assert torch.allclose(weights, torch.tensor(expected_weights), atol=ATOL, rtol=RTOL)
assert torch.allclose(output, torch.tensor(expected_output), atol=ATOL, rtol=RTOL)
assert torch.equal(q, q_before) and torch.equal(k, k_before) and torch.equal(v, v_before)
assert torch.equal(allowed, allowed_before)
assert rejected